In [1]:
!pip install pysentimiento

In [2]:
import pandas as pd
import numpy as np

import re

In [3]:
# se carga el df eda_muestras_etiquetadas
df_muestras = pd.read_csv("../data/labeled/muestras_css_etiquetadas.csv")
df_muestras.head()

,file_name,n_legislatura,cuerpo,fecha,locutor,encabezado,intervencion,n_palabras,n_caracteres,año,sentimiento
0,49_2021-10-13_css.txt,49,css,1634083200000,SEÑORA PRESIDENTA,16) RENDICIÓN DE CUENTAS Y BALANCE DE EJECUCIÓ...,Ya que el artículo 236 tiene que ver con el te...,41,214,2021,Neu
1,49_2021-10-13_css.txt,49,css,1634083200000,SEÑORA PRESIDENTA,16) RENDICIÓN DE CUENTAS Y BALANCE DE EJECUCIÓ...,Se va a votar la moción formulada..\n–15 en 19...,55,313,2021,Neu
2,48_2018-04-03_css.txt,48,css,1522713600000,SEÑOR COUTINHO,"16) LLAMADO A SALA AL MINISTRO DEL INTERIOR, S...",Señora presidenta: estamos convencidos de que ...,373,2217,2018,N+
3,50_2025-09-10_css.txt,50,css,1757462400000,SEÑORA PRESIDENTA,15) EDAD MÍNIMA PARA CONTRAER MATRIMONIO,"Si no se hace uso de la pa labra, se va a vota...",24,114,2025,Neu
4,49_2023-10-11_css.txt,49,css,1696982400000,SEÑORA PRESIDENTA,9) CONECTIVIDAD DE LAS ESCUELAS,Se va a votar el trámite so licitado por el se...,24,139,2023,Neu


Se estudia la distribución de sentimiento para los locutores presidente/presidenta

In [4]:
# Se estudia la distribución de sentimiento de las muestras
df_muestras["sentimiento"].value_counts()

sentimiento
Neu    1667
N       186
P       178
N+      155
P+      114
Name: count, dtype: int64

Se observa que está desbalanceado.

In [5]:
# se filtra los registros que el locutor contiene el texto president y se estudia distribución de sentimiento
df_muestras_president = df_muestras[df_muestras["locutor"].str.contains("president", case=False, na=False)]
df_muestras_president["sentimiento"].value_counts()

sentimiento
Neu    1297
N+       33
P+       18
N        16
P        11
Name: count, dtype: int64

In [6]:
# Se consulta la cantidad de caracteres de la muestra
df_muestras["n_caracteres"].sum()

np.int64(2069368)

In [7]:
# Se consulta la cantidad de caracteres de la muestra filtrada por president
df_muestras_president["n_caracteres"].sum()

np.int64(459640)

In [8]:
# el porcentaje de caracteres de la muestra filtrada por president sobre el total de caracteres de la muestra
(df_muestras_president["n_caracteres"].sum() / df_muestras["n_caracteres"].sum()) * 100

np.float64(22.211612434327776)

Las intevenciones del presidente representan el 61% de la muestra. Sin embargo en cantidad de caracteres sobre el total de la muestra es un 22,2%.
Un 93% de las observaciones que realiza el presidente son neutras.

In [9]:
# se cuentan la cantidad de caracteres por sentimiento en la muestra y su porcentaje sobre el total de caracteres de la muestra
caracteres_por_sentimiento = df_muestras.groupby("sentimiento")["n_caracteres"].sum()
porcentaje_por_sentimiento = (caracteres_por_sentimiento / df_muestras["n_caracteres"].sum()) * 100
porcentaje_por_sentimiento

sentimiento
N      23.636250
N+     12.765153
Neu    27.583929
P      25.563795
P+     10.450872
Name: n_caracteres, dtype: float64

In [10]:
chars_president = df_muestras_president["n_caracteres"].sum()
chars_total = df_muestras["n_caracteres"].sum()
porcentaje = (chars_president / chars_total) * 100

tabla_resumen = pd.DataFrame({
    "Indicador": [
        "Caracteres filtro 'president'",
        "Caracteres totales muestra",
        "Porcentaje sobre el total"
    ],
    "Valor": [
        f"{chars_president:,.0f}",
        f"{chars_total:,.0f}",
        f"{porcentaje:.1f}%"
    ]
})

tabla_resumen


,Indicador,Valor
0,Caracteres filtro 'president',"459,640"
1,Caracteres totales muestra,"2,069,368"
2,Porcentaje sobre el total,22.2%


In [11]:
# se excluyen los registros de df_muestras que contienen president en locutor
df_muestras_sin_president = df_muestras[~df_muestras["locutor"].str.contains("president", case=False, na=False)]
df_muestras_sin_president["sentimiento"].value_counts()

sentimiento
Neu    370
N      170
P      167
N+     122
P+      96
Name: count, dtype: int64

In [12]:
# se cuentan la cantidad de caracteres por sentimiento en la muestra y su porcentaje sobre el total de caracteres de la muestra
caracteres_por_sentimiento = df_muestras_sin_president.groupby("sentimiento")["n_caracteres"].sum()
porcentaje_por_sentimiento = (caracteres_por_sentimiento / df_muestras_sin_president["n_caracteres"].sum()) * 100
porcentaje_por_sentimiento

sentimiento
N      29.359432
N+     15.500321
Neu    10.202034
P      32.404170
P+     12.534043
Name: n_caracteres, dtype: float64

Quitando las intervenciones realizadas por el presidente, se puede observar que las categorías quedan balanceadas.

In [ ]:
# se crea en df_muestras un campo que concatena file_name, n_palabras y n_caracteres para hacer merge con df_limpio
df_muestras_sin_president["merge_key"] = df_muestras_sin_president["file_name"].astype(str) + "_" + df_muestras_sin_president["fecha"].astype(str) + "_" + df_muestras_sin_president["encabezado"].astype(str) + "_" + df_muestras_sin_president["intervencion"].astype(str)

C:\Users\flala\AppData\Local\Temp\ipykernel_1888\101854178.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muestras_sin_president["merge_key"] = df_muestras_sin_president["file_name"].astype(str) + "_" + df_muestras_sin_president["fecha"].astype(str) + "_" + df_muestras_sin_president["encabezado"].astype(str) + "_" + df_muestras_sin_president["intervencion"].astype(str)


In [14]:
# se carga df_limpio 
df_limpio = pd.read_json("../data/processed/df_final_etiquetados.jsonl", orient="records", lines=True)

In [15]:
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23120 entries, 0 to 23119
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   file_name              23120 non-null  object
 1   n_legislatura          23120 non-null  int64 
 2   cuerpo                 23120 non-null  object
 3   fecha                  23120 non-null  int64 
 4   locutor                23120 non-null  object
 5   encabezado             23120 non-null  object
 6   intervencion           23120 non-null  object
 7   n_palabras             23120 non-null  int64 
 8   n_caracteres           23120 non-null  int64 
 9   año                    23120 non-null  int64 
 10  sexo_locutor           23120 non-null  object
 11  apellido_locutor_norm  23120 non-null  object
 12  apellido_key           23120 non-null  object
 13  partido_imputado       23120 non-null  object
 14  metodo_imputacion      23120 non-null  object
 15  score_match        

In [16]:
TOL_PAL = 2
TOL_CHAR = 5
MIN_SIM = 90  # 0-100 (si usás difflib se convierte equivalente)

KEYS = ["file_name", "n_legislatura", "fecha", "locutor"]  # sumá "cuerpo" si ayuda

# --- normalización de texto para comparar intervenciones ---
_ws = re.compile(r"\s+")
_punct = re.compile(r"[^\w\sáéíóúüñÁÉÍÓÚÜÑ]")

def norm_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = _punct.sub(" ", s)
    s = _ws.sub(" ", s)
    return s

# --- Similaridad: rapidfuzz si está, sino difflib ---
try:
    from rapidfuzz.fuzz import token_set_ratio as sim_func
    def similarity(a, b) -> float:
        return float(sim_func(a, b))  # 0..100
except Exception:
    from difflib import SequenceMatcher
    def similarity(a, b) -> float:
        return 100.0 * SequenceMatcher(None, a, b).ratio()

# Preparar dfs mínimos
m = df_muestras[KEYS + ["n_palabras","n_caracteres","intervencion","sentimiento"]].copy()
l = df_limpio[KEYS + ["n_palabras","n_caracteres","intervencion"]].copy()

m["interv_norm"] = m["intervencion"].map(norm_text)
l["interv_norm"] = l["intervencion"].map(norm_text)

# Indexar por bloque
m_groups = {k: g for k, g in m.groupby(KEYS, sort=False)}
l_groups = {k: idx for k, idx in l.groupby(KEYS, sort=False).groups.items()}

sent_out = pd.Series(pd.NA, index=df_limpio.index, dtype="object")
score_out = pd.Series(np.inf, index=df_limpio.index, dtype="float")

common_keys = set(l_groups.keys()) & set(m_groups.keys())

for k in common_keys:
    idxs_l = l_groups[k]
    cand = m_groups[k]

    # Greedy 1-a-1 para evitar que la misma muestra etiquete muchas filas
    used_m = set()

    proposals = []
    for i in idxs_l:
        pal = int(df_limpio.at[i, "n_palabras"])
        cha = int(df_limpio.at[i, "n_caracteres"])
        txt = l.at[i, "interv_norm"]

        # 1) Pre-filtro numérico
        dp = (cand["n_palabras"] - pal).abs()
        dc = (cand["n_caracteres"] - cha).abs()
        ok = (dp <= TOL_PAL) & (dc <= TOL_CHAR)
        if not ok.any():
            continue

        sub = cand.loc[ok]

        # 2) Similaridad por texto dentro de los candidatos numéricamente plausibles
        # (normalmente son pocos; si igual son muchos, MIN_SIM te corta)
        best_j = None
        best_score = np.inf

        for j, row in sub.iterrows():
            sim = similarity(txt, row["interv_norm"])
            if sim < MIN_SIM:
                continue

            # Score combinado:
            # - priorizo que el texto sea MUY parecido
            # - y luego cercanía numérica
            dpj = abs(int(row["n_palabras"]) - pal)
            dcj = abs(int(row["n_caracteres"]) - cha)

            # cuanto más alta similitud, mejor -> (100 - sim)
            score = (100 - sim) * 100 + dpj * 10 + dcj  # pesos ajustables

            if score < best_score:
                best_score = score
                best_j = j

        if best_j is not None:
            proposals.append((best_score, i, best_j))

    # Asignar en orden de mejor score, 1-a-1
    proposals.sort(key=lambda x: x[0])

    for sc, i, j in proposals:
        if j in used_m:
            continue
        if sc < score_out.at[i]:
            sent_out.at[i] = df_muestras.at[j, "sentimiento"]
            score_out.at[i] = sc
            used_m.add(j)

df_limpio["sentimiento"] = sent_out
df_limpio["sent_match_score"] = score_out.replace(np.inf, np.nan)

print("Imputadas:", df_limpio["sentimiento"].notna().sum())


Imputadas: 924


In [33]:
df_limpio

,file_name,n_legislatura,cuerpo,fecha,locutor,encabezado,intervencion,n_palabras,n_caracteres,año,sexo_locutor,apellido_locutor_norm,apellido_key,partido_imputado,metodo_imputacion,score_match,id_intervencion,sentimiento,sent_match_score
0,47_2010-02-15_css.txt,47,css,1266192000000,SEÑOR AMORÍN,DESCONOCIDO,"Sí, prometo. nor desempeñar debidamente el car...",20,119,2010,M,amorin,amorin,COLORADO,exacto,100,1,<NA>,NaN
1,47_2010-02-18_css.txt,47,css,1266451200000,SEÑOR SANABRIA,ORDEN DEL DÍA,"Sí, prometo. –Léase.\nDe mi mayor consideració...",23,123,2010,M,sanabria,sanabria,COLORADO,legislatura_unica_full,95,2,<NA>,NaN
2,47_2010-02-18_css.txt,47,css,1266451200000,SEÑOR LORIER,ORDEN DEL DÍA,Señora Presidenta: solicito una rectificación ...,90,536,2010,M,lorier,lorier,FRENTE AMPLIO,exacto,100,3,<NA>,NaN
3,47_2010-03-03_css.txt,47,css,1267574400000,SEÑOR LACALLE HERRERA,ORDEN DEL DÍA,Señor Presidente: me pareció entender que la a...,58,344,2010,M,lacalle herrera,lacalle,NACIONAL,exacto,100,4,<NA>,NaN
4,47_2010-03-03_css.txt,47,css,1267574400000,SEÑOR LACALLE HERRERA,5) INASISTENCIAS ANTERIORES (SE DA DE LAS SIGU...,"O, por lo menos, entre quienes así lo solicite...",44,275,2010,M,lacalle herrera,lacalle,NACIONAL,exacto,100,5,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23115,50_2025-10-14_css.txt,50,css,1760400000000,SEÑOR OJEDA,28) SUSPENSIÓN DE LA SESIÓN ORDINARIA,Se votó un cuarto intermedio. Lo único que se ...,48,274,2025,M,ojeda,ojeda,COLORADO,exacto,100,23153,<NA>,NaN
23116,50_2025-10-15_css.txt,50,css,1760486400000,SEÑOR SILVA,"7) INASISTENCIAS ANTERIORES DE SENADORES, DESE...",Señora presidenta: retomando la discusión del ...,711,4174,2025,M,silva,silva,COLORADO,exacto,100,23154,<NA>,NaN
23117,50_2025-10-15_css.txt,50,css,1760486400000,SEÑORA BIANCHI,"7) INASISTENCIAS ANTERIORES DE SENADORES, DESE...","Buenos días a todos.\nComo decía, me llama la ...",709,4063,2025,F,bianchi,bianchi,NACIONAL,exacto,100,23155,<NA>,NaN
23118,50_2025-10-15_css.txt,50,css,1760486400000,SEÑOR GLOODTDOFSKY,"7) INASISTENCIAS ANTERIORES DE SENADORES, DESE...","Señor presidente: heme aquí, después de una la...",621,4058,2025,M,gloodtdofsky,gloodtdofsky,FRENTE AMPLIO,exacto,100,23156,<NA>,NaN


In [26]:
# 1) Dropear columnas que ya no se necesitan
cols_drop = ["apellido_key", "metodo_imputacion", "score_match", "sent_match_score"]
df_limpio = df_limpio.drop(columns=[c for c in cols_drop if c in df_limpio.columns])

# 2) Mover id_intervencion al principio
if "id_intervencion" in df_limpio.columns:
    cols = ["id_intervencion"] + [c for c in df_limpio.columns if c != "id_intervencion"]
    df_limpio = df_limpio[cols]


In [27]:
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23120 entries, 0 to 23119
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_intervencion        23120 non-null  int64 
 1   file_name              23120 non-null  object
 2   n_legislatura          23120 non-null  int64 
 3   cuerpo                 23120 non-null  object
 4   fecha                  23120 non-null  int64 
 5   locutor                23120 non-null  object
 6   encabezado             23120 non-null  object
 7   intervencion           23120 non-null  object
 8   n_palabras             23120 non-null  int64 
 9   n_caracteres           23120 non-null  int64 
 10  año                    23120 non-null  int64 
 11  sexo_locutor           23120 non-null  object
 12  apellido_locutor_norm  23120 non-null  object
 13  partido_imputado       23120 non-null  object
 14  sentimiento            924 non-null    object
dtypes: int64(6), object

In [28]:
import numpy as np
import pandas as pd

# 1) Solo registros sin sentimiento
df_pool = df_limpio[df_limpio["sentimiento"].isna()].copy()

# (opcional) reproducibilidad
seed = 1899
rng = np.random.default_rng(seed)

# 2) Bins por n_palabras dentro de cada legislatura (cuantiles)
q = 4

def add_word_bins(g: pd.DataFrame, q=4):
    g = g.copy()
    g["bin_palabras"] = pd.qcut(g["n_palabras"], q=q, duplicates="drop")
    return g

df_pool = (
    df_pool
    .groupby("n_legislatura", group_keys=False)
    .apply(add_word_bins, q=q)
)

# Si alguna legislatura quedó con bin_palabras NaN (caso extremo), la mandamos a un bin único
df_pool["bin_palabras"] = df_pool["bin_palabras"].astype("object").fillna("bin_unico")

# 3) Asignación de cupos por estrato: (legislatura x bin_palabras) proporcional a su tamaño
target_n = 300

counts = (
    df_pool
    .groupby(["n_legislatura", "bin_palabras"])
    .size()
    .rename("N")
    .reset_index()
)

counts["quota"] = counts["N"] / counts["N"].sum() * target_n
counts["take"]  = np.floor(counts["quota"]).astype(int)

# no pedir más de lo que hay
counts["take"] = counts[["take", "N"]].min(axis=1)

# repartir faltantes por mayor decimal (y donde aún haya disponibles)
missing = target_n - counts["take"].sum()

if missing > 0:
    counts["frac"] = counts["quota"] - np.floor(counts["quota"])
    counts["remaining"] = counts["N"] - counts["take"]

    candidates = counts[counts["remaining"] > 0].sort_values("frac", ascending=False)
    idx = candidates.index.to_list()

    for i in range(missing):
        counts.loc[idx[i % len(idx)], "take"] += 1

# 4) Sampleo final respetando cupos
samples = []
for (leg, binp), sub in df_pool.groupby(["n_legislatura", "bin_palabras"]):
    n_take = int(
        counts.loc[
            (counts["n_legislatura"] == leg) & (counts["bin_palabras"] == binp),
            "take"
        ].iloc[0]
    )
    if n_take > 0:
        samples.append(sub.sample(n=n_take, random_state=seed))

df_muestra_300 = pd.concat(samples, ignore_index=True)

# Check
print(df_muestra_300.shape)
print(df_muestra_300["n_legislatura"].value_counts())
print(pd.cut(df_muestra_300["n_palabras"], bins=10).value_counts().sort_index())

# >>> CON TODOS LOS CAMPOS <<<
df_muestra_300_etiquetar = df_muestra_300.sort_values(["n_legislatura", "n_palabras"]).reset_index(drop=True)


C:\Users\flala\AppData\Local\Temp\ipykernel_5992\881218927.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_word_bins, q=q)
C:\Users\flala\AppData\Local\Temp\ipykernel_5992\881218927.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pool["bin_palabras"] = df_pool["bin_palabras"].astype("object").fillna("bin_unico")


(300, 16)
n_legislatura
47    106
49    100
48     82
50     12
Name: count, dtype: int64
n_palabras
(11.279, 488.1]     220
(488.1, 960.2]       53
(960.2, 1432.3]      13
(1432.3, 1904.4]      6
(1904.4, 2376.5]      1
(2376.5, 2848.6]      3
(2848.6, 3320.7]      1
(3320.7, 3792.8]      1
(3792.8, 4264.9]      0
(4264.9, 4737.0]      2
Name: count, dtype: int64


In [29]:
# 1) Leer el excel etiquetado
ruta_xlsx = "../data/labeled/muestra_300_etiquetadas.xlsx"   # ajustá si está en otra carpeta
df_etq = pd.read_excel(ruta_xlsx)

# 2) Elegí qué columnas querés traer (mínimo sentimiento)
cols_a_traer = ["id_intervencion", "sentimiento"]
df_etq = df_etq[cols_a_traer].copy()

# 3) Asegurar mismo tipo de id
df_limpio["id_intervencion"] = pd.to_numeric(df_limpio["id_intervencion"], errors="coerce")
df_etq["id_intervencion"]     = pd.to_numeric(df_etq["id_intervencion"], errors="coerce")

# 4) Evitar duplicar filas si en el excel hay ids repetidos
df_etq = df_etq.dropna(subset=["id_intervencion"]).drop_duplicates("id_intervencion", keep="last")

# 5) Merge (left): trae etiquetas al df_limpio
df_limpio = df_limpio.merge(df_etq, on="id_intervencion", how="left", suffixes=("", "_etq"))

# 6) Si df_limpio ya tenía 'sentimiento' y querés SOLO completar los NaN con lo del excel:
if "sentimiento_etq" in df_limpio.columns:
    df_limpio["sentimiento"] = df_limpio["sentimiento"].combine_first(df_limpio["sentimiento_etq"])
    df_limpio = df_limpio.drop(columns=["sentimiento_etq"], errors="ignore")

In [30]:
#se exporta a excel para etiquetar manualmente
df_muestra_300_etiquetar.to_excel("../data/labeled/muestra_300_para_etiquetar.xlsx", index=False)

# EDA datos de muestra 

In [31]:
df_limpio["sentimiento"] = df_limpio["sentimiento"].replace({"N+": "N", "P+": "P"})


In [32]:
df_limpio_muestra = df_limpio[df_limpio["sentimiento"].notna()].copy()


In [33]:
# Se crea una variable binaria que indica si el locutor es parte del oficialismo o no
# Las legistaturas 47, 48 y 50 el partido oficialista es el Frente Amplio. Para la legistaura 49, el oficialismo esta compuesto por el Partido Nacional, Partido Colorado, Cabildo Abierto y Partido Independiente.
df_limpio_muestra["oficialismo"] = np.where(
    ((df_limpio_muestra["n_legislatura"].isin([47, 48, 50])) & (df_limpio_muestra["partido_imputado"] == "FRENTE AMPLIO")) | 
    ((df_limpio_muestra["n_legislatura"] == 49) & (df_limpio_muestra["partido_imputado"].isin(["NACIONAL", "COLORADO", "CABILDO ABIERTO", "INDEPENDIENTE"])) )
, 1, 0)
df_limpio_muestra["oficialismo"].value_counts()

oficialismo
0    616
1    608
Name: count, dtype: int64

## Distribución por Legislatura

In [34]:
# cantidad de intervenciones por legislatura
df_limpio_muestra["n_legislatura"].value_counts()

n_legislatura
49    422
47    418
48    317
50     67
Name: count, dtype: int64

In [35]:
# Cantidad de intervenciones por partido
df_limpio_muestra["partido_imputado"].value_counts()

partido_imputado
FRENTE AMPLIO      564
NACIONAL           406
COLORADO           208
CABILDO ABIERTO     27
INDEPENDIENTE       19
Name: count, dtype: int64

In [22]:
# distribución de intervenciones de partidos por cada legislatura
pd.crosstab(df_limpio_muestra["n_legislatura"], df_limpio_muestra["partido_imputado"])

partido_imputado,CABILDO ABIERTO,COLORADO,FRENTE AMPLIO,INDEPENDIENTE,NACIONAL
n_legislatura,,,,,
47,0,94,193,0,131
48,0,65,149,19,84
49,27,34,189,0,172
50,0,15,33,0,19


In [36]:
# Cantidad de intervenciones por partido
df_limpio_muestra["sexo_locutor"].value_counts()

sexo_locutor
M    982
F    242
Name: count, dtype: int64

In [37]:
# distribución de intervenciones de partidos por cada legislatura
pd.crosstab(df_limpio_muestra["n_legislatura"], df_limpio_muestra["sexo_locutor"])

sexo_locutor,F,M
n_legislatura,,
47,19,399
48,66,251
49,141,281
50,16,51


In [38]:
ct = pd.crosstab(
    df_limpio_muestra["n_legislatura"],
    [df_limpio_muestra["partido_imputado"], df_limpio_muestra["sexo_locutor"]],
    dropna=False
)
ct

partido_imputado CABILDO ABIERTO     COLORADO     FRENTE AMPLIO       \
sexo_locutor                   F   M        F   M             F    M   
n_legislatura                                                          
47                             0   0        0  94            19  174   
48                             0   0        2  63            52   97   
49                             2  25       13  21            70  119   
50                             0   0        0  15             6   27   

partido_imputado INDEPENDIENTE     NACIONAL       
sexo_locutor                 F   M        F    M  
n_legislatura                                     
47                           0   0        0  131  
48                           0  19       12   72  
49                           0   0       56  116  
50                           0   0       10    9

In [39]:
# # distribución de intervenciones de partidos por cada legislatura
pd.crosstab(df_limpio_muestra["n_legislatura"], df_limpio_muestra["oficialismo"])

oficialismo,0,1
n_legislatura,,
47,225,193
48,168,149
49,189,233
50,34,33


In [27]:
# distribucion de intervenciones por oficialismo y sentimiento
pd.crosstab(df_limpio_muestra["oficialismo"], df_limpio_muestra["sentimiento"], dropna=False)

sentimiento,N,Neu,P
oficialismo,,,
0,275,184,157
1,135,268,205


In [ ]:
# distribucion de intervenciones por partido y sentimiento
pd.crosstab(df_limpio_muestra["partido_imputado"], df_limpio_muestra["sentimiento"], dropna=False)

sentimiento,N,Neu,P
partido_imputado,,,
CABILDO ABIERTO,11,7,9
COLORADO,81,65,62
FRENTE AMPLIO,161,226,177
INDEPENDIENTE,6,8,5
NACIONAL,151,146,109


In [29]:
df_limpio["sentimiento"].value_counts()

sentimiento
Neu    452
N      410
P      362
Name: count, dtype: int64

Se exporta para utilizar dataset con los modelos

In [ ]:
df_limpio.to_json("../data/processed/df_final_para_modelo.jsonl", orient="records", lines=True)